In [64]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import time


In [3]:
# Load the dataset split files
train_df = pd.read_csv(r"datasets\DAIC-Woz\train_split_Depression_AVEC2017.csv")
dev_df = pd.read_csv(r"datasets\DAIC-Woz\dev_split_Depression_AVEC2017.csv")

In [10]:
display(train_df)

,Participant_ID,PHQ8_Binary,PHQ8_Score,Gender,PHQ8_NoInterest,PHQ8_Depressed,PHQ8_Sleep,PHQ8_Tired,PHQ8_Appetite,PHQ8_Failure,PHQ8_Concentrating,PHQ8_Moving
0,303,0,0,0,0,0,0.0,0,0,0,0,0
1,304,0,6,0,0,1,1.0,2,2,0,0,0
2,305,0,7,1,0,1,1.0,2,2,1,0,0
3,310,0,4,1,1,1,0.0,0,0,1,1,0
4,312,0,2,1,0,0,1.0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
102,485,0,2,1,0,1,0.0,0,0,0,0,1
103,486,0,4,0,1,1,0.0,1,0,1,0,0
104,487,0,0,0,0,0,0.0,0,0,0,0,0
105,488,0,0,0,0,0,0.0,0,0,0,0,0


In [29]:
# Function to load AU features
def load_au_features(participant_id, data_path):
    au_file = os.path.join(data_path, f"{participant_id}_P", f"{participant_id}_CLNF_AUs.txt")
    au_data = pd.read_csv(au_file)
    # Drop the first four columns (frame, timestamp, confidence, success)
    au_data = au_data.iloc[:, 4:]
    # Average the AUs across all frames
    au_mean = au_data.mean(axis=0)
    return au_mean

In [87]:
def load_additional_features(participant_id, data_path, feature_type):
    feature_file = os.path.join(data_path, f'{participant_id}_P', f'{participant_id}_{feature_type}.txt')
    feature_data = pd.read_csv(feature_file)
    # Drop the first four columns (frame, timestamp, confidence, success)
    feature_data = feature_data.iloc[:, 4:]
    # Average the features across all frames
    feature_mean = feature_data.mean(axis=0)
    return feature_mean

In [95]:
# Function to load all features
def load_all_features(participant_id, data_path):
    au_features = load_au_features(participant_id, data_path)
    pose_features = load_additional_features(participant_id, data_path, 'CLNF_pose')
    gaze_features = load_additional_features(participant_id, data_path, 'CLNF_gaze')
    #hog_features = load_additional_features(participant_id, data_path, 'CLNF_hog')
    
    # Concatenate all features into a single feature vector
    all_features = pd.concat([au_features, pose_features, gaze_features], axis=0)
    return all_features

In [30]:
# Path to the dataset
data_path = r"datasets\DAIC-Woz"


In [24]:
data_path = r"datasets\DAIC-Woz"
participant_id = 300


au_file = os.path.join(data_path, f"{participant_id}_P", f"{participant_id}_CLNF_AUs.txt")

print(au_file)
print(os.path.isfile(au_file))

datasets\DAIC-Woz\300_P\300_CLNF_AUs.txt
True


In [96]:
# Extract features and labels for the training set
train_features = []
train_labels = []

In [97]:
for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    participant_id = int(row["Participant_ID"])
    label = int(row["PHQ8_Binary"])
    try:
        features = load_all_features(participant_id, data_path)
        train_features.append(features)
        train_labels.append(label)
    except FileNotFoundError:
        print(f"File not found for participant {participant_id}")

100%|██████████| 107/107 [00:42<00:00,  2.55it/s]


In [98]:
train_features = np.array(train_features)
train_labels = np.array(train_labels)

In [99]:
print(train_labels)

[0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 1 0 0 0 1 0 0 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1
 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 1 0
 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [100]:
# Train-Test Split
X_train, X_val, y_train, y_val = train_test_split(train_features, train_labels, test_size=0.2, random_state=42)


In [101]:
print(y_train)
print(y_val)

[0 0 1 1 1 0 1 0 0 0 0 1 0 0 1 1 0 1 0 0 1 0 1 1 0 0 0 0 1 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 1 0
 0 0 1 0 0 0 0 0 1 1 0]
[0 1 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 1 1 0 1 1]


In [102]:
# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [103]:
# Define the parameter grid for GridSearchCV
param_grid = {
    'learning_rate': [0.1, 0.01],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'n_estimators': [100, 200, 300]
}

In [104]:
# Initialize the classifier
xgb_clf = xgb.XGBClassifier(random_state=42)


In [105]:
# Instantiate GridSearchCV
grid_search = GridSearchCV(estimator=xgb_clf, param_grid=param_grid, scoring='accuracy', cv=5, verbose=1)

# Fit GridSearchCV
start_time = time.time()
grid_search.fit(X_resampled, y_resampled)
end_time = time.time()

# Get the best estimator
best_clf = grid_search.best_estimator_

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


In [106]:
# Calculate the duration
training_time = end_time - start_time

print(f"Time taken to fit the XGBoost model: {training_time:.2f} seconds")

Time taken to fit the XGBoost model: 440.70 seconds


In [107]:
# Predictions using the best model
y_pred = best_clf.predict(X_val)

In [108]:
# Evaluate the best model
accuracy = accuracy_score(y_val, y_pred)
report = classification_report(y_val, y_pred)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Accuracy: {accuracy}')
print(f'Classification Report:\n{report}')

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.8}
Accuracy: 0.5454545454545454
Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.79      0.69        14
           1       0.25      0.12      0.17         8

    accuracy                           0.55        22
   macro avg       0.43      0.46      0.43        22
weighted avg       0.48      0.55      0.50        22



# TEMP

In [ ]:
def load_visual_features(data_directory, gaze_confidence_threshold=0.90):
    participants_data = []

    # Iterate over each participant"s folder
    for folder_name in os.listdir(data_directory):
        if folder_name.endswith("_P"):
            participant_id = folder_name[:-2]  # Extract participant ID

            # Load gaze data
            gaze_file = os.path.join(data_directory, folder_name, f"{participant_id}_CLNF_gaze.txt")
            gaze_data = pd.read_csv(gaze_file, header=None, names=["frame", "timestamp", "confidence", "detection_success",
                                                                   "x0", "y0", "z0", "x1", "y1", "z1",
                                                                   "x_h0", "y_h0", "z_h0", "x_h1", "y_h1", "z_h1"])

            # Filter frames with low confidence
            gaze_data = gaze_data[(gaze_data["detection_success"] == 1) & (gaze_data["confidence"] >= gaze_confidence_threshold)]

            # Aggregate gaze direction vectors
            gaze_vectors = gaze_data[["x0", "y0", "z0", "x1", "y1", "z1", "x_h0", "y_h0", "z_h0", "x_h1", "y_h1", "z_h1"]].mean().values
            participants_data.append((participant_id, gaze_vectors))

    return participants_data